# Proseg segmentation for decoded ISS transcripts

This notebook runs the Rust-based Proseg executable and converts its consensus cell polygons to the same SciPy sparse `.npz` mask used by the rest of the ISS pipeline. Proseg needs an initial cell/nucleus prior. Pass an existing label mask when possible; otherwise the wrapper derives one from DAPI with Cellpose.

Install Proseg once with `conda create -n proseg -c conda-forge -c bioconda rust-proseg=3.2.0` (no Rust toolchain needed), or use `cargo install proseg`. Restart the notebook kernel if the executable is not yet on `PATH`; when it is in a separate environment, pass `proseg_command=['conda', 'run', '-n', 'proseg', 'proseg']`.

In [ ]:
from pathlib import Path
import ISS_postprocessing.segmentation as SEG

## Inputs

The decoded ISS defaults are `target`, `xc`, and `yc`. Coordinates must refer to the same stitched image and are assumed to be pixels. Set the physical image pixel size accurately because Proseg works in microns.

In [ ]:
input_dir = Path('/path/to/regions')
region = 'R1'
pixel_size_um = 0.325

transcripts_file = input_dir / region / 'decoding' / '2_decoded' / f'{region}_decoded.csv'
dapi_file = input_dir / region / 'preprocessing' / 'Cycle1' / '3_stitched' / 'Cycle1_ch4.tif'

# Recommended when an existing Cellpose/StarDist mask is available.
# Accepted formats: sparse .npz, .npy, TIFF, or a NumPy array.
initial_mask = None

In [ ]:
preview = SEG.normalize_transcript_table(
    transcripts_file,
    gene_col='target',
    x_col='xc',
    y_col='yc',
    # quality_col='quality_minimum',
    # min_quality=0.5,
)
preview.head(), preview[['x', 'y']].describe()

## Run Proseg

Method-native outputs are retained in `R1_proseg_work`. The final file is `R1_proseg_stitched_expanded.npz`. Proseg is stochastic, so results can differ slightly between runs.

In [ ]:
labels, labels_coo = SEG.proseg_segmentation(
    transcripts=transcripts_file,
    image=dapi_file,
    initial_mask=initial_mask,
    region=region,
    input_dir=input_dir,
    gene_col='target',
    x_col='xc',
    y_col='yc',
    pixel_size_um=pixel_size_um,
    nthreads=16,
    seed_expansion_distance=20,
    extra_args=[
        '--voxel-size', '1.0',
        '--burnin-voxel-size', '4.0',
        '--samples', '200',
        '--burnin-samples', '200',
    ],
    overwrite=False,
)
labels.shape, int(labels.max()), labels_coo.nnz

## Inspect the result

In [ ]:
SEG.inspect_and_work_with_segmentation(
    input_dir=input_dir,
    region=region,
    segmentation_method='proseg',
    input_image_type='stitched',
    DAPI_ch=4,
    crop_coords=None,
)